## 1. 라이브러리 로드

In [58]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import platform
import ast
from collections import Counter, defaultdict
import json

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

---
## 2. 데이터 로드

In [ ]:
df_platinum_Match = pd.read_csv('./tft_game_dataset/TFT_Platinum_MatchData.csv')
df_diamond_Match = pd.read_csv('./tft_game_dataset/TFT_Diamond_MatchData.csv')
df_master_Match = pd.read_csv('./tft_game_dataset/TFT_Master_MatchData.csv')
df_grand_master_Match = pd.read_csv('./tft_game_dataset/TFT_GrandMaster_MatchData.csv')
df_challenger_Match = pd.read_csv('./tft_game_dataset/TFT_Challenger_MatchData.csv')

df_champion_info = pd.read_csv('./tft_game_dataset/TFT_Champion_CurrentVersion.csv')
df_items_info = pd.read_csv('./tft_game_dataset/TFT_Item_CurrentVersion.csv')

In [60]:
# 매치관련 데이터가 담긴 딕셔너리 정의
match_data = {
    'platinum': df_platinum_Match,
    'diamond': df_diamond_Match,
    'master': df_master_Match,
    'grand_master': df_grand_master_Match,
    'challenger': df_challenger_Match,
}

# 각 티어별 테이블에 'Tier'정보가 담긴 컬럼 추가
for name, df in match_data.items():
    df['tier'] = name


# 모든 티어의 경기데이터가 담긴 데이터프레임 제작
# ignore_index=True: 데이터를 병합한 뒤, 새로운 인덱스 부여
df_all_match = pd.concat(match_data.values(), ignore_index=True)

df_all_match.head(3)

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
0,KR_4291707834,1963.905273,6,27,5,1390.165771,"{'Cybernetic': 1, 'Demolitionist': 1, 'Infiltrator': 1, 'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Celestial': 1, 'Set3_Void': 1, 'Sniper': 1}","{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",platinum
1,KR_4291707834,1963.905273,8,37,3,1891.282715,"{'Blaster': 1, 'Chrono': 1, 'Cybernetic': 4, 'Demolitionist': 1, 'Rebel': 1, 'Set3_Blademaster': 2, 'Set3_Brawler': 1, 'Set3_Sorcerer': 1, 'Set3_Void': 1, 'Valkyrie': 1, 'Vanguard': 2}","{'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}",platinum
2,KR_4291707834,1963.905273,6,25,7,1279.461060,"{'Blaster': 1, 'Cybernetic': 1, 'DarkStar': 2, 'Infiltrator': 1, 'Mercenary': 1, 'Set3_Blademaster': 1, 'Set3_Mystic': 1, 'Valkyrie': 1}","{'Fiora': {'items': [1], 'star': 1}, 'Shaco': {'items': [6], 'star': 1}, 'Karma': {'items': [4], 'star': 1}, 'MissFortune': {'items': [3], 'star': 1}}",platinum


---
## 3. 데이터 전처리
### 3-1. 중복행 제거

In [61]:
# 중복행 제거
duplicates = df_all_match[df_all_match.duplicated(keep=False)]

print(f"중복 제거 전: {len(df_all_match)}")
print(f"중복된 행 개수: {len(duplicates)}")

# 결과 확인
df_all_match = df_all_match.drop_duplicates()
print(f"\n중복 제거 후: {len(df_all_match)}")

중복 제거 전: 399998
중복된 행 개수: 80

중복 제거 후: 399930


### 3-2. 컬럼명 소문자로 통일

In [62]:
# 전체 컬럼명 소문자 통일
df_all_match.columns = df_all_match.columns.str.lower()

print(df_all_match.columns)

Index(['gameid', 'gameduration', 'level', 'lastround', 'ranked',
       'ingameduration', 'combination', 'champion', 'tier'],
      dtype='str')


### 3-3. ranked=0인 데이터 삭제

In [63]:
# ranked = 0이 포함된 경기 데이터 삭제
df_all_match_2 = df_all_match.copy()

# ranked==0인 행의 gameId 추출
drop_game_ids = df_all_match_2[df_all_match_2['ranked']==0]['gameid'].unique()

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx)

print(f"ranked = 0인 데이터가 포함된 경기 데이터 삭제하기 전: {df_all_match.shape}")
print(f"ranked = 0인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

ranked = 0인 데이터가 포함된 경기 데이터 삭제하기 전: (399930, 9)
ranked = 0인 데이터가 포함된 경기 데이터 삭제한 후: (399886, 9)


### 3-4. 경기시간 = 0인 데이터 삭제

In [64]:
# 전체 게임시간 = 0인 행의 GameId 추출
zero_duration_ids = set(df_all_match_2[df_all_match_2['gameduration'] == 0]['gameid'])

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_2 = df_all_match_2[df_all_match_2['gameid'].isin(zero_duration_ids)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_2)

print(f"gameduration = 0인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

gameduration = 0인 데이터가 포함된 경기 데이터 삭제한 후: (399886, 9)


### 3-5. 경기 참여 유저 수가 8명 미만인 게임 삭제

In [65]:
# 게임 ID별 플레이어 수 정보 추가
df_all_match_2['player_cnt'] = df_all_match_2['gameid'].map(df_all_match_2['gameid'].value_counts())

# player_cnt < 8인 행의 gameId 추출
drop_game_ids_3 = df_all_match_2[df_all_match_2['player_cnt'] < 8]['gameid'].unique()

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_3 = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids_3)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_3)

print(f"player_cnt < 8인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

player_cnt < 8인 데이터가 포함된 경기 데이터 삭제한 후: (399872, 10)


### 3-6. 시즌2 데이터 삭제

In [66]:
# 시즌 2 고유 시너지 목록
season2_keys = {'Alchemist', 'Avatar', 'Berserker',
                'Crystal', 'Desert', 'Druid', 'Electric',
                'Light', 'Mage', 'Mountain', 'Mystic', 'Ocean',
                'Poison', 'Predator', 'Set2_Assassin', 'Set2_Blademaster',
                'Set2_Glacial', 'Set2_Ranger', 'Shadow', 'Soulbound',
                'Summoner', 'Warden', 'Wind', 'Woodland'}  # 시즌 2 시너지 입력


# 시즌 2 시너지가 하나라도 포함되면 시즌 2로 분류
df_all_match_2['season'] = df_all_match_2['combination'].apply(
    lambda x: 'season 2' if any(k in season2_keys for k in ast.literal_eval(x).keys())
    else 'season 3'
)

# season 2인 행의 gameId 추출
drop_game_ids_4 = df_all_match_2[df_all_match_2['season'] == 'season 2']['gameid'].unique()


# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_4 = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids_4)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_4)

print(f"season 2인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

season 2인 데이터가 포함된 경기 데이터 삭제한 후: (396640, 11)


### 3-7. 시즌 3 시너지 더미 데이터 ”TemplateTrait” 삭제
- 시너지 데이터 중 해당 값인 `{'TemplateTrait': 1}`만 삭제

In [67]:
# TemplateTrait 키가 있는 행만 필터링 후 값 확인
df_all_match_2[df_all_match_2['combination'].apply(
    lambda x: 'TemplateTrait' in ast.literal_eval(x).keys() # TemplateTrait 키가 있는 행만 필터링
)]['combination'].apply(                                    # 필터링된 행 중에서 combination 컬럼만 가져옴
    lambda x: ast.literal_eval(x).get('TemplateTrait')      # TemplateTrait 키에 할당된 값을 가져옴
).value_counts()


combination
1    65987
Name: count, dtype: int64

In [68]:
# 분석 목적에 영향을 주지 않는 dummy 데이터로 판단하여 시너지 중 'TemplateTrait'만 삭제
df_all_match_2['combination'] = df_all_match_2['combination'].apply(
    lambda x: json.dumps(
        # key: value = 키-값의 쌍을 의미함
        {key: value for key, value in ast.literal_eval(x).items() if key != 'TemplateTrait'},
        ensure_ascii=False
    )
)

In [69]:
# TemplateTrait 키가 있는 행 수 확인 → (0,11)이면 완전히 삭제된 것
df_all_match_2[df_all_match_2['combination'].apply(
    lambda x: 'TemplateTrait' in ast.literal_eval(x).keys()
)].shape

(0, 11)

### 3-8. 유저 ID 컬럼 제작

In [70]:
df_all_match_2['user_id'] = 'KR-USER-' + (df_all_match_2.index + 1).astype(str)

In [71]:
df_all_match_2['user_id']

0              KR-USER-1
1              KR-USER-2
2              KR-USER-3
3              KR-USER-4
4              KR-USER-5
               ...      
399993    KR-USER-399994
399994    KR-USER-399995
399995    KR-USER-399996
399996    KR-USER-399997
399997    KR-USER-399998
Name: user_id, Length: 396640, dtype: str

---
## 3-9. 빈 딕셔너리 (결측 combination/champion) 감지 및 분석

### 문제 
- `gameId` 단위로 정상 데이터를 처리하는 과정에서, 정원 8명 중 일부 플레이어의 `combination`/`champion`이 빈 딕셔너리 `{}`로 기록되어 있음
- 한 게임의 8명 중 한 명이라도 문제가 있으면 전체 게임의 신뢰도 감소

### 처리 규칙
1. **둘 다 결측 (Combo='{}' AND Champ='{}')**: 해당 행이 포함된 게임 전체 삭제
2. **하나만 결측**:
   - `flag_1` (boolean): Combo O + Champ X → 사후검정용
   - `flag_2` (boolean): Combo X + Champ O → 복구용


In [74]:
# 딕셔너리 파싱 함수
def parse_dict(val):
    """문자열을 딕셔너리로 파싱"""
    try:
        if isinstance(val, str):
            return ast.literal_eval(val)
        return val
    except:
        return {}

In [75]:
def is_empty_dict(x):
    """딕셔너리가 비어있는지 확인"""
    try:
        parsed = parse_dict(x)
        return isinstance(parsed, dict) and len(parsed) == 0
    except:
        return False

print("함수 정의 완료")

함수 정의 완료


In [76]:
print("처리 전 상태 확인")
print(f"\n 데이터 기본 정보")
print(f"전체 행 수: {len(df_all_match_2):,}")
print(f"전체 게임 수: {df_all_match_2['gameid'].nunique():,}")

# 빈 딕셔너리 감지
empty_combo_count = 0
empty_champ_count = 0
both_empty_count = 0

for idx, row in df_all_match_2.iterrows():
    is_combo_empty = is_empty_dict(row['combination'])
    is_champ_empty = is_empty_dict(row['champion'])
    
    if is_combo_empty:
        empty_combo_count += 1
    if is_champ_empty:
        empty_champ_count += 1
    if is_combo_empty and is_champ_empty:
        both_empty_count += 1

print(f"\n결측 패턴")
print(f"Empty combination: {empty_combo_count:,} ({empty_combo_count/len(df_all_match_2)*100:.2f}%)")
print(f"Empty champion: {empty_champ_count:,} ({empty_champ_count/len(df_all_match_2)*100:.2f}%)")
print(f"둘 다 empty: {both_empty_count:,} ({both_empty_count/len(df_all_match_2)*100:.2f}%)")
print(f"둘 중 하나만 empty: {empty_combo_count + empty_champ_count - 2*both_empty_count:,}")

처리 전 상태 확인

 데이터 기본 정보
전체 행 수: 396,640
전체 게임 수: 49,561

결측 패턴
Empty combination: 380 (0.10%)
Empty champion: 298 (0.08%)
둘 다 empty: 263 (0.07%)
둘 중 하나만 empty: 152


In [77]:
print("둘 다 결측인 행이 포함된 게임 찾기")

games_to_delete = set()

for idx, row in df_all_match_2.iterrows():
    is_combo_empty = is_empty_dict(row['combination'])
    is_champ_empty = is_empty_dict(row['champion'])
    
    if is_combo_empty and is_champ_empty:
        games_to_delete.add(row['gameid'])

print(f"\n삭제 대상 게임")
print(f"둘 다 empty 행이 포함된 게임: {len(games_to_delete):,}개")


둘 다 결측인 행이 포함된 게임 찾기

삭제 대상 게임
둘 다 empty 행이 포함된 게임: 217개


In [ ]:
# 플래그 컬럼 생성

# 플래그 정의
# flag_1 (boolean): combination O + champion X (사후검정용)
# flag_2 (boolean): combination X + champion O (복구용)

df_all_match_2['flag_1'] = False  # combination O + champion X
df_all_match_2['flag_2'] = False  # combination X + champion O

flag_1_count = 0
flag_2_count = 0

for idx, row in df_all_match_2.iterrows():
    is_combo_empty = is_empty_dict(row['combination'])
    is_champ_empty = is_empty_dict(row['champion'])
    
    # flag_1: combination O + champion X 
    if not is_combo_empty and is_champ_empty:
        df_all_match_2.at[idx, 'flag_1'] = True
        flag_1_count += 1
    
    # flag_2: combination X + champion O
    if is_combo_empty and not is_champ_empty:
        df_all_match_2.at[idx, 'flag_2'] = True
        flag_2_count += 1

print("\n플래그 컬럼 생성 결과")
print(f"flag_1 (Combo O + Champ X): {flag_1_count:,}행 ({flag_1_count/len(df_all_match_2)*100:.2f}%)")
print(f"flag_2 (Combo X + Champ O): {flag_2_count:,}행 ({flag_2_count/len(df_all_match_2)*100:.2f}%)")
print(f"둘 다 False: {len(df_all_match_2) - flag_1_count - flag_2_count:,}행")

플래그 컬럼 생성

플래그 정의
flag_1 (boolean): combination O + champion X (사후검정용)
flag_2 (boolean): combination X + champion O (복구용)

플래그 컬럼 생성 결과
flag_1 (Combo O + Champ X): 35행 (0.01%)
flag_2 (Combo X + Champ O): 117행 (0.03%)
둘 다 False: 396,488행


In [ ]:
print("둘 다 결측인 행이 포함된 게임 삭제")

rows_before = len(df_all_match_2)
games_before = df_all_match_2['gameid'].nunique()

# 게임 삭제
df_all_match_3 = df_all_match_2[~df_all_match_2['gameid'].isin(games_to_delete)].reset_index(drop=True)

#확인용
rows_after = len(df_all_match_3)
games_after = df_all_match_3['gameid'].nunique()

print("\n삭제 결과")
print(f"삭제 전 행 수: {rows_before:,}")
print(f"삭제 전 게임 수: {games_before:,}")
print(f"\n삭제된 행 수: {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.2f}%)")
print(f"삭제된 게임 수: {len(games_to_delete):,} ({len(games_to_delete)/games_before*100:.2f}%)")
print(f"\n삭제 후 행 수: {rows_after:,}")
print(f"삭제 후 게임 수: {games_after:,}")
print(f"데이터 보존율: {rows_after/rows_before*100:.2f}%")

둘 다 결측인 행이 포함된 게임 삭제

[삭제 결과
삭제 전 행 수: 396,640
삭제 전 게임 수: 49,561

삭제된 행 수: 1,736 (0.44%)
삭제된 게임 수: 217 (0.44%)

삭제 후 행 수: 394,904
삭제 후 게임 수: 49,344
데이터 보존율: 99.56%


In [82]:
df_all_match_3

,gameid,gameduration,level,lastround,ranked,ingameduration,combination,champion,tier,player_cnt,season,user_id,flag_1,flag_2
0,KR_4291707834,1963.905273,6,27,5,1390.165771,"{""Cybernetic"": 1, ""Demolitionist"": 1, ""Infiltrator"": 1, ""Rebel"": 1, ""Set3_Brawler"": 1, ""Set3_Celestial"": 1, ""Set3_Void"": 1, ""Sniper"": 1}","{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",platinum,8,season 3,KR-USER-1,False,False
1,KR_4291707834,1963.905273,8,37,3,1891.282715,"{""Blaster"": 1, ""Chrono"": 1, ""Cybernetic"": 4, ""Demolitionist"": 1, ""Rebel"": 1, ""Set3_Blademaster"": 2, ""Set3_Brawler"": 1, ""Set3_Sorcerer"": 1, ""Set3_Void"": 1, ""Valkyrie"": 1, ""Vanguard"": 2}","{'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}",platinum,8,season 3,KR-USER-2,False,False
2,KR_4291707834,1963.905273,6,25,7,1279.461060,"{""Blaster"": 1, ""Cybernetic"": 1, ""DarkStar"": 2, ""Infiltrator"": 1, ""Mercenary"": 1, ""Set3_Blademaster"": 1, ""Set3_Mystic"": 1, ""Valkyrie"": 1}","{'Fiora': {'items': [1], 'star': 1}, 'Shaco': {'items': [6], 'star': 1}, 'Karma': {'items': [4], 'star': 1}, 'MissFortune': {'items': [3], 'star': 1}}",platinum,8,season 3,KR-USER-3,False,False
3,KR_4291707834,1963.905273,7,38,2,1955.608521,"{""DarkStar"": 1, ""Protector"": 2, ""Set3_Blademaster"": 1, ""Set3_Celestial"": 5, ""Set3_Mystic"": 1, ""Sniper"": 1, ""StarGuardian"": 2, ""Vanguard"": 2}","{'Poppy': {'items': [], 'star': 2}, 'Xayah': {'items': [19, 23, 19], 'star': 3}, 'Rakan': {'items': [], 'star': 2}, 'XinZhao': {'items': [16], 'star': 2}, 'Mordekaiser': {'items': [35, 67, 33], 'star': 3}, 'Ashe': {'items': [], 'star': 2}, 'Soraka': {'items': [68, 47], 'star': 2}}",platinum,8,season 3,KR-USER-4,False,False
4,KR_4291707834,1963.905273,8,38,1,1955.608521,"{""Blaster"": 1, ""Chrono"": 5, ""DarkStar"": 3, ""Protector"": 1, ""Set3_Blademaster"": 1, ""Set3_Brawler"": 1, ""Set3_Sorcerer"": 2, ""Sniper"": 2}","{'TwistedFate': {'items': [36, 27], 'star': 3}, 'Caitlyn': {'items': [49, 29], 'star': 2}, 'JarvanIV': {'items': [56], 'star': 2}, 'Blitzcrank': {'items': [15], 'star': 2}, 'Shen': {'items': [77, 6], 'star': 2}, 'Ezreal': {'items': [16], 'star': 2}, 'Lux': {'items': [], 'star': 2}, 'Jhin': {'items': [], 'star': 2}}",platinum,8,season 3,KR-USER-5,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394899,KR_4357265434,2094.518555,7,33,4,1838.332764,"{""DarkStar"": 2, ""Demolitionist"": 2, ""Infiltrator"": 4, ""MechPilot"": 3, ""Set3_Sorcerer"": 2, ""Set3_Void"": 1, ""Valkyrie"": 1}","{'KhaZix': {'items': [67, 26, 56], 'star': 2}, 'KaiSa': {'items': [38, 44, 33], 'star': 2}, 'Annie': {'items': [], 'star': 3}, 'Shaco': {'items': [45, 29, 16], 'star': 3}, 'Rumble': {'items': [55, 66, 77], 'star': 2}, 'Lux': {'items': [46], 'star': 2}, 'Fizz': {'items': [], 'star': 2}}",challenger,8,season 3,KR-USER-399994,False,False
394900,KR_4357265434,2094.518555,8,33,5,1837.548096,"{""Blaster"": 1, ""Chrono"": 2, ""Cybernetic"": 6, ""Infiltrator"": 2, ""ManaReaver"": 2, ""Set3_Blademaster"": 3, ""Set3_Brawler"": 1, ""Vanguard"": 1}","{'Fiora': {'items': [35], 'star': 2}, 'Leona': {'items': [22], 'star': 1}, 'Shen': {'items': [], 'star': 1}, 'Lucian': {'items': [57, 2, 23], 'star': 2}, 'Vi': {'items': [67, 36], 'star': 2}, 'Irelia': {'items': [29, 28, 15], 'star': 2}, 'Thresh': {'items': [57], 'star': 2}, 'Ekko': {'items': [35, 15, 2], 'star': 2}}",challenger,8,season 3,KR-USER-399995,False,False
394901,KR_4357265434,2094.518555,9,33,6,1833.472046,"{""Chrono"": 2, ""Cybernetic"": 1, ""Demolitionist"": 2, ""Infiltrator"": 2, ""MechPilot"": 1, ""Mercenary"": 1, ""Rebel"": 1, ""Set3_Brawler"": 4, ""Set3_Sorcerer"": 2, ""Set3_Voi

In [ ]:
# Flag 1, 2 게임 ID 리스트 추출

# 플래그 확인
flag_1_mask = df_all_match_3['flag_1'] == True
flag_2_mask = df_all_match_3['flag_2'] == True

# 게임 ID 추출
flag_1_game_ids = set(df_all_match_3[flag_1_mask]['gameid'].unique())
flag_2_game_ids = set(df_all_match_3[flag_2_mask]['gameid'].unique())

print(f"\nFlag 게임 ID 개수")
print(f"flag_1 게임 ID 개수: {len(flag_1_game_ids):,}개")
print(f"flag_2 게임 ID 개수: {len(flag_2_game_ids):,}개")
print(f"두 플래그가 겹치는 게임: {len(flag_1_game_ids & flag_2_game_ids):,}개")

# 리스트로 변환
flag_1_game_ids_list = sorted(list(flag_1_game_ids))
flag_2_game_ids_list = sorted(list(flag_2_game_ids))


Flag 1, 2 게임 ID 리스트 추출

Flag 게임 ID 개수
flag_1 게임 ID 개수: 5개
flag_2 게임 ID 개수: 115개
두 플래그가 겹치는 게임: 0개


---
## 4. 티어가 섞인 경기에 대한 처리

In [92]:
# 티어가 섞인 gameId 추출
mixed_gameids = df_all_match_2.groupby('gameid')['tier'].nunique() # gameid별로 tier의 고유값 개수를 계산
mixed_gameids = mixed_gameids[mixed_gameids > 1].index # 고유값이 2 이상인 gameId만 추출

# 해당 gameId의 티어 구성 확인
# mixed_gameids에 포함된 gameId를 가진 행만 추출 -> gameid별로 어떤 티어가 몇 명씩 있는지 카운트
df_all_match_2[df_all_match_2['gameid'].isin(mixed_gameids)].groupby('gameid')['tier'].value_counts()

gameid         tier        
KR_4263820118  platinum        8
               master          8
KR_4313697214  master          8
               platinum        8
KR_4320079059  platinum        8
               diamond         8
KR_4344513862  master          8
               diamond         8
KR_4347884427  diamond         8
               master          8
KR_4357966612  grand_master    8
               platinum        8
KR_4358922415  master          8
               diamond         8
KR_4361594426  master          8
               diamond         8
KR_4361773461  diamond         8
               grand_master    8
KR_4362846604  master          8
               diamond         8
KR_4364453473  grand_master    8
               diamond         8
KR_4365284161  master          8
               diamond         8
KR_4378896137  diamond         8
               platinum        8
KR_4381231118  diamond         8
               platinum        8
KR_4387025966  diamond         8
               

In [93]:
# 티어 순서 정의 (숫자가 높을수록 높은 티어)
tier_order = {
    'platinum': 1,
    'diamond': 2,
    'master': 3,
    'grand_master': 4,
    'challenger': 5
}

In [94]:
# 기본값 normal로 설정
df_all_match_2['mix_flag'] = 'normal'

# gameId별 최고/최저 티어 추출
highest_tier = {}
lowest_tier = {}

for gameid, group in df_all_match_2[df_all_match_2['gameid'].isin(mixed_gameids)].groupby('gameid')['tier']:
    tier_numbers = [tier_order[t] for t in group]
    
    max_tier_number = max(tier_numbers)
    min_tier_number = min(tier_numbers)
    
    for t in tier_order:
        if tier_order[t] == max_tier_number:
            highest_tier[gameid] = t
        if tier_order[t] == min_tier_number:
            lowest_tier[gameid] = t

# 믹스매치 행에 high/low 플래그 부여
for idx, row in df_all_match_2[df_all_match_2['gameid'].isin(mixed_gameids)].iterrows():
    if row['tier'] == highest_tier[row['gameid']]:
        df_all_match_2.loc[idx, 'mix_flag'] = 'high'
    else:
        df_all_match_2.loc[idx, 'mix_flag'] = 'low'

# 확인
df_all_match_2['mix_flag'].value_counts()

mix_flag
normal    396336
low          152
high         152
Name: count, dtype: int64

### 4-1.동일 게임 ID를 기준으로 여러 티어가 섞여있는 게임이라면, 상위 티어를 가진 유저의 정보를 보존함

In [109]:
df_all_match_3_high = df_all_match_2[df_all_match_2['mix_flag'].isin(['normal', 'high'])]

In [110]:
df_all_match_3_high['mix_flag'].value_counts()

mix_flag
normal    396336
high         152
Name: count, dtype: int64

### 4-2.동일 게임 ID를 기준으로 여러 티어가 섞여있는 게임이라면, 하위 티어를 가진 유저의 정보를 보존함

In [111]:
df_all_match_3_low = df_all_match_2[df_all_match_2['mix_flag'].isin(['normal', 'low'])]

In [ ]:
df_all_match_3_low['mix_flag'].value_counts()

mix_flag
normal    396336
low          152
Name: count, dtype: int64

In [112]:
df_all_match_3_high

,gameid,gameduration,level,lastround,ranked,ingameduration,combination,champion,tier,player_cnt,season,user_id,flag_1,flag_2,mix_flag
0,KR_4291707834,1963.905273,6,27,5,1390.165771,"{""Cybernetic"": 1, ""Demolitionist"": 1, ""Infiltrator"": 1, ""Rebel"": 1, ""Set3_Brawler"": 1, ""Set3_Celestial"": 1, ""Set3_Void"": 1, ""Sniper"": 1}","{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",platinum,8,season 3,KR-USER-1,False,False,normal
1,KR_4291707834,1963.905273,8,37,3,1891.282715,"{""Blaster"": 1, ""Chrono"": 1, ""Cybernetic"": 4, ""Demolitionist"": 1, ""Rebel"": 1, ""Set3_Blademaster"": 2, ""Set3_Brawler"": 1, ""Set3_Sorcerer"": 1, ""Set3_Void"": 1, ""Valkyrie"": 1, ""Vanguard"": 2}","{'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}",platinum,8,season 3,KR-USER-2,False,False,normal
2,KR_4291707834,1963.905273,6,25,7,1279.461060,"{""Blaster"": 1, ""Cybernetic"": 1, ""DarkStar"": 2, ""Infiltrator"": 1, ""Mercenary"": 1, ""Set3_Blademaster"": 1, ""Set3_Mystic"": 1, ""Valkyrie"": 1}","{'Fiora': {'items': [1], 'star': 1}, 'Shaco': {'items': [6], 'star': 1}, 'Karma': {'items': [4], 'star': 1}, 'MissFortune': {'items': [3], 'star': 1}}",platinum,8,season 3,KR-USER-3,False,False,normal
3,KR_4291707834,1963.905273,7,38,2,1955.608521,"{""DarkStar"": 1, ""Protector"": 2, ""Set3_Blademaster"": 1, ""Set3_Celestial"": 5, ""Set3_Mystic"": 1, ""Sniper"": 1, ""StarGuardian"": 2, ""Vanguard"": 2}","{'Poppy': {'items': [], 'star': 2}, 'Xayah': {'items': [19, 23, 19], 'star': 3}, 'Rakan': {'items': [], 'star': 2}, 'XinZhao': {'items': [16], 'star': 2}, 'Mordekaiser': {'items': [35, 67, 33], 'star': 3}, 'Ashe': {'items': [], 'star': 2}, 'Soraka': {'items': [68, 47], 'star': 2}}",platinum,8,season 3,KR-USER-4,False,False,normal
4,KR_4291707834,1963.905273,8,38,1,1955.608521,"{""Blaster"": 1, ""Chrono"": 5, ""DarkStar"": 3, ""Protector"": 1, ""Set3_Blademaster"": 1, ""Set3_Brawler"": 1, ""Set3_Sorcerer"": 2, ""Sniper"": 2}","{'TwistedFate': {'items': [36, 27], 'star': 3}, 'Caitlyn': {'items': [49, 29], 'star': 2}, 'JarvanIV': {'items': [56], 'star': 2}, 'Blitzcrank': {'items': [15], 'star': 2}, 'Shen': {'items': [77, 6], 'star': 2}, 'Ezreal': {'items': [16], 'star': 2}, 'Lux': {'items': [], 'star': 2}, 'Jhin': {'items': [], 'star': 2}}",platinum,8,season 3,KR-USER-5,False,False,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399993,KR_4357265434,2094.518555,7,33,4,1838.332764,"{""DarkStar"": 2, ""Demolitionist"": 2, ""Infiltrator"": 4, ""MechPilot"": 3, ""Set3_Sorcerer"": 2, ""Set3_Void"": 1, ""Valkyrie"": 1}","{'KhaZix': {'items': [67, 26, 56], 'star': 2}, 'KaiSa': {'items': [38, 44, 33], 'star': 2}, 'Annie': {'items': [], 'star': 3}, 'Shaco': {'items': [45, 29, 16], 'star': 3}, 'Rumble': {'items': [55, 66, 77], 'star': 2}, 'Lux': {'items': [46], 'star': 2}, 'Fizz': {'items': [], 'star': 2}}",challenger,8,season 3,KR-USER-399994,False,False,normal
399994,KR_4357265434,2094.518555,8,33,5,1837.548096,"{""Blaster"": 1, ""Chrono"": 2, ""Cybernetic"": 6, ""Infiltrator"": 2, ""ManaReaver"": 2, ""Set3_Blademaster"": 3, ""Set3_Brawler"": 1, ""Vanguard"": 1}","{'Fiora': {'items': [35], 'star': 2}, 'Leona': {'items': [22], 'star': 1}, 'Shen': {'items': [], 'star': 1}, 'Lucian': {'items': [57, 2, 23], 'star': 2}, 'Vi': {'items': [67, 36], 'star': 2}, 'Irelia': {'items': [29, 28, 15], 'star': 2}, 'Thresh': {'items': [57], 'star': 2}, 'Ekko': {'items': [35, 15, 2], 'star': 2}}",challenger,8,season 3,KR-USER-399995,False,False,normal
399995,KR_4357265434,2094.518555,9,33,6,1833.472046,"{""Chrono"": 2, ""Cybernetic"": 1, ""Demolitionist"": 2, ""Infiltrator"": 2, ""MechPilot"": 1, ""Mercenary"": 1, ""Reb

In [114]:
df_all_match_3_high.to_csv('./preprocess_trial_csv/version1_상위티어게임 편입 전처리(복구전).csv', index=False, encoding='utf-8-sig')

In [115]:
df_all_match_3_low.to_csv('./preprocess_trial_csv/version1_하위티어게임 편입 전처리(복구전).csv', index=False, encoding='utf-8-sig')

---
## 5.챔피언+아이템 정보를 활용하여 콤비네이션 컬럼 채우기

In [113]:
# 챔피언별 시너지 딕셔너리

champion_traits = {
    'Ahri': ['StarGuardian', 'Set3_Sorcerer'],
    'Annie': ['MechPilot', 'Set3_Sorcerer'],
    'Ashe': ['Set3_Celestial', 'Sniper'],
    'AurelionSol': ['Rebel', 'Starship'],
    'Blitzcrank': ['Chrono', 'Set3_Brawler'],
    'Caitlyn': ['Chrono', 'Sniper'],
    'ChoGath': ['Set3_Void', 'Set3_Brawler'],
    'Darius': ['SpacePirate', 'ManaReaver'],
    'Ekko': ['Cybernetic', 'Infiltrator'],
    'Ezreal': ['Chrono', 'Blaster'],
    'Fiora': ['Cybernetic', 'Set3_Blademaster'],
    'Fizz': ['MechPilot', 'Infiltrator'],
    'Gangplank': ['SpacePirate', 'Demolitionist', 'Mercenary'],
    'Graves': ['SpacePirate', 'Blaster'],
    'Irelia': ['Cybernetic', 'Set3_Blademaster', 'ManaReaver'],
    'JarvanIV': ['DarkStar', 'Protector'],
    'Jayce': ['SpacePirate', 'Vanguard'],
    'Jhin': ['DarkStar', 'Sniper'],
    'Jinx': ['Rebel', 'Blaster'],
    'KaiSa': ['Valkyrie', 'Infiltrator'],
    'Karma': ['DarkStar', 'Set3_Mystic'],
    'Kassadin': ['Set3_Celestial', 'ManaReaver'],
    'Kayle': ['Valkyrie', 'Set3_Blademaster'],
    'KhaZix': ['Set3_Void', 'Infiltrator'],
    'Leona': ['Cybernetic', 'Vanguard'],
    'Lucian': ['Cybernetic', 'Blaster'],
    'Lulu': ['Set3_Celestial', 'Set3_Mystic'],
    'Lux': ['DarkStar', 'Set3_Sorcerer'],
    'Malphite': ['Rebel', 'Set3_Brawler'],
    'MasterYi': ['Rebel', 'Set3_Blademaster'],
    'MissFortune': ['Valkyrie', 'Blaster', 'Mercenary'],
    'Mordekaiser': ['DarkStar', 'Vanguard'],
    'Neeko': ['StarGuardian', 'Protector'],
    'Poppy': ['StarGuardian', 'Vanguard'],
    'Rakan': ['Set3_Celestial', 'Protector'],
    'Rumble': ['MechPilot', 'Demolitionist'],
    'Shaco': ['DarkStar', 'Infiltrator'],
    'Shen': ['Chrono', 'Set3_Blademaster'],
    'Sona': ['Rebel', 'Set3_Mystic'],
    'Soraka': ['StarGuardian', 'Set3_Mystic'],
    'Syndra': ['StarGuardian', 'Set3_Sorcerer'],
    'Thresh': ['Chrono', 'ManaReaver'],
    'TwistedFate': ['Chrono', 'Set3_Sorcerer'],
    'VelKoz': ['Set3_Void', 'Set3_Sorcerer'],
    'Vi': ['Cybernetic', 'Set3_Brawler'],
    'WuKong': ['Chrono', 'Vanguard'],
    'Xayah': ['Set3_Celestial', 'Set3_Blademaster'],
    'Xerath': ['DarkStar', 'Set3_Sorcerer'],
    'XinZhao': ['Set3_Celestial', 'Protector'],
    'Yasuo': ['Rebel', 'Set3_Blademaster'],
    'Ziggs': ['Rebel', 'Demolitionist'],
    'Zoe': ['StarGuardian', 'Set3_Sorcerer']
}

#뒤집개템
item_to_trait = {
    18: 'Set3_Blademaster',   # Blade of the Ruined King
    28: 'Infiltrator',        # Infiltrator's Talons
    38: 'Demolitionist',      # Demolitionist's Charge
    48: 'StarGuardian',       # Star Guardian's Charm
    58: 'Rebel',              # Rebel Medal
    68: 'Set3_Celestial',     # Celestial Orb
    78: 'Protector',          # Protector's Chestguard
    89: 'DarkStar'            # Dark Star's Heart
}



In [116]:
df_all_match_3_high[df_all_match_3_high['flag_2']]

,gameid,gameduration,level,lastround,ranked,ingameduration,combination,champion,tier,player_cnt,season,user_id,flag_1,flag_2,mix_flag
6627,KR_4271274609,2037.501343,6,13,8,637.688232,{},"{'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'KaiSa': {'items': [], 'star': 1}, 'Mordekaiser': {'items': [], 'star': 2}, 'Kassadin': {'items': [], 'star': 1}}",platinum,8,season 3,KR-USER-6628,False,True,normal
10005,KR_4359182660,2064.300293,5,13,8,621.877319,{},"{'JarvanIV': {'items': [], 'star': 1}, 'Lux': {'items': [], 'star': 1}}",platinum,8,season 3,KR-USER-10006,False,True,normal
15897,KR_4320179268,2129.208252,4,12,8,604.033447,{},"{'Malphite': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 1}, 'Mordekaiser': {'items': [], 'star': 1}, 'Ashe': {'items': [], 'star': 1}}",platinum,8,season 3,KR-USER-15898,False,True,normal
20909,KR_4380989360,2041.039062,8,37,2,2032.800903,{},"{'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [], 'star': 2}, 'Irelia': {'items': [16, 19, 29], 'star': 2}, 'WuKong': {'items': [], 'star': 2}, 'Thresh': {'items': [], 'star': 1}, 'Ekko': {'items': [], 'star': 2}}",platinum,8,season 3,KR-USER-20910,False,True,normal
22014,KR_4366152411,1980.549683,7,24,8,1259.300781,{},"{'Malphite': {'items': [], 'star': 2}, 'KhaZix': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 3}, 'Mordekaiser': {'items': [], 'star': 2}, 'Shaco': {'items': [], 'star': 2}, 'Jayce': {'items': [], 'star': 1}, 'WuKong': {'items': [], 'star': 2}}",platinum,8,season 3,KR-USER-22015,False,True,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
382016,KR_4345849647,2204.278564,8,35,3,1932.822144,{},"{'Ziggs': {'items': [], 'star': 2}, 'Malphite': {'items': [], 'star': 2}, 'Yasuo': {'items': [], 'star': 2}, 'Sona': {'items': [], 'star': 2}, 'MasterYi': {'items': [], 'star': 2}, 'Jinx': {'items': [], 'star': 2}, 'Kayle': {'items': [], 'star': 1}, 'MissFortune': {'items': [], 'star': 2}}",challenger,8,season 3,KR-USER-382017,False,True,normal
387453,KR_4348875448,2091.441895,6,24,8,1340.589844,{},"{'JarvanIV': {'items': [], 'star': 3}, 'Xayah': {'items': [], 'star': 2}, 'Shen': {'items': [], 'star': 2}, 'Rakan': {'items': [], 'star': 2}, 'XinZhao': {'items': [], 'star': 2}, 'Kassadin': {'items': [], 'star': 2}}",challenger,8,season 3,KR-USER-387454,False,True,normal
395452,KR_4354056563,1920.435181,7,17,7,868.813721,{},"{'Graves': {'items': [], 'star': 2}, 'Blitzcrank': {'items': [], 'star': 1}, 'Darius': {'items': [], 'star': 2}, 'Ezreal': {'items': [], 'star': 1}, 'Kassadin': {'items': [], 'star': 1}, 'Vi': {'items': [], 'star': 1}, 'Kayle': {'items': [], 'star': 1}}",challenger,8,season 3,KR-USER-395453,False,True,normal
396637,KR_4355031855,2115.783936,6,21,8,1145.362305,{},"{'TwistedFate': {'items': [], 'star': 2}, 'Malphite': {'items': [], 'star': 3}, 'KhaZix': {'items': [], 'star': 3}, 'Blitzcrank': {'items': [], 'star': 2}, 'Vi': {'items': [], 'star': 2}, 'ChoGath': {'items': [], 'star': 1}}",challenger,8,season 3,KR-USER-396638,False,True,normal


In [118]:
# 복구할 데이터를 정리하는 함수 정의
def make_combination_from_champion_and_items(champion_dict):
    # 시너지 카운트 저장용 딕셔너리
    trait_count = {}
    
    # 챔피언 이름, 정보 하나씩 반복
    for champ_name, champ_info in champion_dict.items():
        # 챔피언 기본 시너지 조회
        base_traits = champion_traits.get(champ_name, [])

        # 기본 시너지 개수 누적
        for trait in base_traits:
            trait_count[trait] = trait_count.get(trait, 0) + 1
        
        # 장착 아이템 목록 조회
        item_ids = champ_info.get('items', [])
        
        # 아이템이 추가 시너지를 주는지 확인
        for item_id in item_ids:
            added_trait = item_to_trait.get(item_id)

            # 추가 시너지가 있으면 개수 누적
            if added_trait:
                trait_count[added_trait] = trait_count.get(added_trait, 0) + 1
    
    # 시너지별 개수 반환
    return trait_count

In [119]:
# 비어있는 combination 데이터를 일부 복구하는 함수
def rebuild_combination(row):
    # flag_2가 1인 경우 (combination은 비어있고 champion 데이터가 있는 경우)
    # champion 데이터를 기반으로 combination을 재구성
    
    if row['flag_2'] == 1: # 행 단위로 가져와서 검사하고 실행
        return make_combination_from_champion_and_items(ast.literal_eval(row['champion']))
    
    # flag_2가 0인 경우 (combination 데이터가 정상적으로 있는 경우)
    # 기존 combination 값을 그대로 유지
    else:
        return row['combination']

### 5-1. df_all_match_3_high 데이터프레임 데이터 복구
- 믹스 티어매치 경기, 상위 티어로 정보 편입하여 보존한 데이터프레임

In [120]:
# 함수를 행 단위로 적용하여 'combination_rebuild' 컬럼의 값을 채울 예정
df_all_match_3_high['combination_rebuild'] = df_all_match_3_high.apply(rebuild_combination, axis=1)

In [121]:
# 데이터 복구 결과 확인
df_all_match_3_high[df_all_match_3_high['flag_2'] == 1][['combination', 'combination_rebuild']].head()

,combination,combination_rebuild
6627,{},"{'Cybernetic': 2, 'Set3_Blademaster': 1, 'Vanguard': 2, 'Valkyrie': 1, 'Infiltrator': 1, 'DarkStar': 1, 'Set3_Celestial': 1, 'ManaReaver': 1}"
10005,{},"{'DarkStar': 2, 'Protector': 1, 'Set3_Sorcerer': 1}"
15897,{},"{'Rebel': 1, 'Set3_Brawler': 1, 'Cybernetic': 1, 'Vanguard': 2, 'DarkStar': 1, 'Set3_Celestial': 1, 'Sniper': 1}"
20909,{},"{'Cybernetic': 6, 'Set3_Blademaster': 2, 'Vanguard': 2, 'Blaster': 1, 'Set3_Brawler': 1, 'ManaReaver': 2, 'Chrono': 2, 'Infiltrator': 1}"
22014,{},"{'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Void': 1, 'Infiltrator': 2, 'Cybernetic': 1, 'Vanguard': 4, 'DarkStar': 2, 'SpacePirate': 1, 'Chrono': 1}"


### 5-2. df_all_match_3_low 데이터프레임 데이터 복구
- 믹스 티어매치 경기, 하위 티어로 정보 편입하여 보존한 데이터프레임

In [122]:
# 함수를 행 단위로 적용하여 'combination_rebuild' 컬럼의 값을 채울 예정
df_all_match_3_low['combination_rebuild'] = df_all_match_3_low.apply(rebuild_combination, axis=1)

In [123]:
# 데이터 복구 결과 확인
df_all_match_3_low[df_all_match_3_low['flag_2'] == 1][['combination', 'combination_rebuild']].head()

,combination,combination_rebuild
6627,{},"{'Cybernetic': 2, 'Set3_Blademaster': 1, 'Vanguard': 2, 'Valkyrie': 1, 'Infiltrator': 1, 'DarkStar': 1, 'Set3_Celestial': 1, 'ManaReaver': 1}"
10005,{},"{'DarkStar': 2, 'Protector': 1, 'Set3_Sorcerer': 1}"
15897,{},"{'Rebel': 1, 'Set3_Brawler': 1, 'Cybernetic': 1, 'Vanguard': 2, 'DarkStar': 1, 'Set3_Celestial': 1, 'Sniper': 1}"
20909,{},"{'Cybernetic': 6, 'Set3_Blademaster': 2, 'Vanguard': 2, 'Blaster': 1, 'Set3_Brawler': 1, 'ManaReaver': 2, 'Chrono': 2, 'Infiltrator': 1}"
22014,{},"{'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Void': 1, 'Infiltrator': 2, 'Cybernetic': 1, 'Vanguard': 4, 'DarkStar': 2, 'SpacePirate': 1, 'Chrono': 1}"


---
## 6. 데이터 복구 결과 확인, csv 파일로 저장

In [124]:
# combination_rebuild에 빈 딕셔너리가 있는지 확인
df_all_match_3_high['combination_rebuild'].apply(
    lambda x: x == {} if isinstance(x, dict) else ast.literal_eval(x) == {}
).sum()

np.int64(263)

In [125]:
# combination_rebuild에 빈 딕셔너리가 있는지 확인
df_all_match_3_low['combination_rebuild'].apply(
    lambda x: x == {} if isinstance(x, dict) else ast.literal_eval(x) == {}
).sum()

np.int64(263)

In [ ]:
# # combination 데이터를 복구한 뒤, csv 파일로 저장
# df_all_match_3_high.to_csv("./게임단위_게임데이터_상위랭커보존-데이터복구완료.csv", index=False, encoding='utf-8-sig')
# df_all_match_3_low.to_csv("./게임단위_게임데이터_하위랭커보존-데이터복구완료.csv", index=False, encoding='utf-8-sig')